# 04 — Avaliação e Interpretação

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

RANDOM_STATE = 42
CAMINHO_DADOS = Path('..') / 'data' / 'raw' / 'WineQT.csv'
TARGET = 'qualidade_do_vinho'
pd.set_option('display.max_columns', None)
PROCESSED = Path('..') / 'data' / 'processed'
MODELS = Path('..') / 'results' / 'models'

In [ ]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")

Recriamos a mesma divisão treino/teste do notebook 03 (mesmo `RANDOM_STATE`) e carregamos os modelos salvos, para avaliar exatamente os mesmos modelos no mesmo conjunto de teste.

In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste: {X_test.shape[0]} amostras')

In [ ]:
import joblib

grids_padronizados = joblib.load(MODELS / "grids_padronizados.joblib")

Para a comparação final usamos os modelos da última etapa (GridSearchCV com dados padronizados).

## 1. Métricas no conjunto de teste

In [ ]:
modelo_regressao_logistica = grids_padronizados['Regressão Logística'].best_estimator_
arvore_decisao = grids_padronizados['Árvore de Decisão'].best_estimator_
arvore_random = grids_padronizados['Random Forest'].best_estimator_

y_predito_regre = modelo_regressao_logistica.predict(X_test)
y_predito_arvore = arvore_decisao.predict(X_test)
y_predito_arvore_random = arvore_random.predict(X_test)

predicoes = {
    'Regressão Logística': y_predito_regre,
    'Árvore de Decisão': y_predito_arvore,
    'Random Forest': y_predito_arvore_random
}

In [ ]:
print('=== Regressão Logística ===')
print(f'Taxa de acerto: {accuracy_score(y_test, y_predito_regre):.2f}')
print(classification_report(y_test, y_predito_regre, target_names=['Baixo/Médio', 'Alto']))

In [ ]:
print('=== Árvore de Decisão ===')
print(f'Taxa de acerto: {accuracy_score(y_test, y_predito_arvore):.2f}')
print(classification_report(y_test, y_predito_arvore, target_names=['Baixo/Médio', 'Alto']))

In [ ]:
print('=== Random Forest ===')
print(f'Taxa de acerto: {accuracy_score(y_test, y_predito_arvore_random):.2f}')
print(classification_report(y_test, y_predito_arvore_random, target_names=['Baixo/Médio', 'Alto']))

In [ ]:
print('Acurácia predizendo sempre Baixo/Médio:', round((y_test == 0).mean(), 2))

## 2. Justificativa das métricas

Como a classe Alto é pequena, olhamos principalmente para a classe Alto:
- Precisão: dos vinhos que o modelo disse que são Alto, quantos realmente são.
- Recall: dos vinhos que realmente são Alto, quantos o modelo encontrou.
- F1: equilíbrio entre precisão e recall.

Predizendo sempre Baixo/Médio já se tem 87% de acurácia no teste, então a acurácia sozinha não mostra se o modelo é bom.

**Qual erro custa mais caro:** olhando o cenário da empresa, seria prejuízo classificar um vinho Alto como Baixo/Médio (falso negativo), porque um lote com potencial premium seria tratado como comum. O falso positivo (vinho comum marcado como Alto) custa menos, porque o lote ainda passa pela prova sensorial antes de ir para a linha premium. Por isso damos peso ao recall da classe Alto, sem ignorar a precisão, e usamos o F1 e a AUC para comparar os modelos.

## 3. Matriz de confusão e curva ROC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20,5))
for ax, (nome, y_predito) in zip(axes, predicoes.items()):
    sns.heatmap(confusion_matrix(y_test, y_predito), annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Baixo/Médio', 'Alto'], yticklabels=['Baixo/Médio', 'Alto'])
    ax.set_title(nome)
    ax.set_xlabel('Predição')
    ax.set_ylabel('Dados Reais')
plt.tight_layout()
plt.show()

In [ ]:
probabilidades = {
    'Regressão Logística': modelo_regressao_logistica.predict_proba(X_test)[:, 1],
    'Árvore de Decisão': arvore_decisao.predict_proba(X_test)[:, 1],
    'Random Forest': arvore_random.predict_proba(X_test)[:, 1]
}

plt.figure(figsize=(8,6))
for nome, prob in probabilidades.items():
    fpr, tpr, limites = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f'{nome} (AUC = {roc_auc_score(y_test, prob):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatório')
plt.title('Curva ROC')
plt.xlabel('Taxa de falsos positivos')
plt.ylabel('Taxa de verdadeiros positivos')
plt.legend()
plt.show()

Pontos principais da comparação:
- Predizendo sempre Baixo/Médio já se tem 87% de acurácia no teste, então a acurácia sozinha não mostra se o modelo é bom. Por isso comparamos pelo F1, precisão e recall da classe Alto.
- A Regressão Logística e a Árvore de Decisão encontraram 21 dos 27 vinhos Alto (recall de 0,78), e o Random Forest encontrou 20 (recall de 0,74). Olhando o cenário da empresa, seria prejuízo classificar um vinho Alto como Baixo.
- A diferença maior está nos falsos positivos, ou seja, vinhos Baixo/Médio marcados como Alto: 21 no Random Forest, 32 na Árvore de Decisão e 37 na Regressão Logística.
- O Random Forest teve o melhor resultado: F1 de 0,59, AUC de 0,92 e o melhor F1 na validação cruzada (0,54). Com precisão de 0,49, de cada 2 vinhos que ele aponta como Alto, 1 realmente é.
- A Regressão Logística teve AUC de 0,88, mas precisão de 0,36: de cada 3 vinhos que ela aponta como Alto, só 1 realmente é.
- A Árvore de Decisão ficou no meio no F1 (0,52), mas teve a menor AUC (0,79).

O Random Forest foi o melhor tanto na validação cruzada quanto no teste, então é o modelo que escolhemos. Ele encontra um vinho Alto a menos que os outros dois, mas erra bem menos nos vinhos comuns.

O teste tem só 27 vinhos Alto, então diferenças de 1 ou 2 vinhos entre os modelos estão dentro da variação esperada. Por isso a validação cruzada é importante para confirmar a escolha.

## 4. Importância das variáveis

In [ ]:
importancia = pd.Series(arvore_random.named_steps['modelo'].feature_importances_, index=X.columns).sort_values()
ax = importancia.plot(kind='barh', figsize=(8,6))
ax.bar_label(ax.containers[0], fmt='%.3f')
plt.title('Importância das variáveis no Random Forest')
plt.show()

In [ ]:
coeficientes = pd.Series(modelo_regressao_logistica.named_steps['modelo'].coef_[0], index=X.columns).sort_values()
ax = coeficientes.plot(kind='barh', figsize=(8,6))
ax.bar_label(ax.containers[0], fmt='%.2f')
plt.title('Coeficientes da Regressão Logística')
plt.show()

No Random Forest, as variáveis que mais se destacaram foram alcohol, sulphates e volatile acidity. Elas também foram as que apresentaram diferenças mais evidentes entre as categorias durante a análise exploratória. Em seguida, aparecem citric acid e density, que também apresenta uma relação com o teor alcoólico.

Na Regressão Logística, os coeficientes indicam como cada variável está relacionada à probabilidade de um vinho ser classificado como Alto. Coeficientes positivos favorecem essa classificação, enquanto os negativos estão associados a uma menor probabilidade. Nesse caso, alcohol (+0,85) e sulphates (+0,74) apresentam os maiores valores positivos. Já volatile acidity (-0,58) está entre as variáveis com maior associação negativa. Esses resultados são semelhantes aos encontrados no Random Forest.

É importante considerar, porém, que algumas variáveis possuem relações entre si. Isso acontece, por exemplo, com density e alcohol, com as variáveis relacionadas à acidez e com as medidas de SO2. Por esse motivo, os coeficientes da Regressão Logística não devem ser interpretados isoladamente. A variável density, por exemplo, apresenta um coeficiente de -0,77, mas parte dessa relação pode estar associada à sua correlação com o álcool.

## 5. Implicações práticas

### Possíveis aplicações na produção

Os resultados permitem levantar algumas hipóteses sobre como as características físico-químicas podem ser utilizadas no acompanhamento da produção. Essas interpretações são baseadas nas relações encontradas nos dados e não comprovam uma relação de causa e efeito.

Álcool: o teor alcoólico pode estar relacionado ao estágio de maturação das uvas e ao andamento da fermentação. Por isso, acompanhar esses fatores pode ajudar a entender melhor as características associadas aos vinhos de maior qualidade.

Acidez volátil: foi uma das variáveis com maior associação negativa com a qualidade. Como está relacionada principalmente à presença de ácido acético, seu monitoramento pode ajudar a identificar possíveis alterações durante a produção. Cuidados com a higiene, a exposição ao oxigênio e o controle da temperatura também podem contribuir para evitar problemas.

Sulfatos: os vinhos classificados como Alto apresentaram valores maiores dessa variável. Como os sulfatos estão relacionados à proteção do vinho contra alterações indesejadas, acompanhar seus níveis pode ser útil durante a produção, respeitando as características e os limites adequados para cada vinho.

### Aplicação prática do modelo

Os resultados obtidos indicam que o Random Forest consegue identificar aproximadamente 7 em cada 10 vinhos da categoria Alto. Por outro lado, cerca de metade dos vinhos classificados pelo modelo como Alto realmente pertence a essa categoria. Isso mostra que o modelo pode ser útil para uma primeira seleção dos lotes, mas ainda apresenta limitações nas previsões.

Na prática, a ferramenta poderia ajudar a vinícola a definir quais lotes devem ser avaliados primeiro por meio da prova sensorial. Dessa forma, as análises de laboratório serviriam como apoio à tomada de decisão, sem substituir a avaliação dos especialistas.

## 6. Limitações

Por fim, é importante destacar que as correlações encontradas não comprovam que uma variável seja responsável por aumentar a qualidade do vinho. Um teor alcoólico mais elevado, por exemplo, não significa necessariamente que o vinho será melhor. O objetivo do modelo é identificar padrões presentes nos dados para auxiliar na classificação dos lotes.

Outras limitações:
- **Base pequena:** são 1.018 vinhos após remover duplicatas, e só 27 vinhos Alto no teste. Um ou dois vinhos a mais ou a menos já mudam bastante as métricas.
- **Precisão ainda baixa:** cerca de metade dos vinhos que o Random Forest aponta como Alto não é. O modelo serve para priorizar lotes, não para substituir a prova sensorial.
- **Escopo da base:** os dados são de vinhos tintos de uma única variedade (Vinho Verde português). O modelo não deve ser usado para outros tipos de vinho sem novos dados.
- **Nota subjetiva:** a variável alvo vem da avaliação de especialistas, que pode variar entre provadores.

Com mais tempo ou mais dados, faríamos:
- Ajustar o limite de probabilidade a partir do qual o modelo diz "Alto", de acordo com o custo de cada tipo de erro para a vinícola (hoje usamos o padrão de 0,5).
- Testar outros modelos (como XGBoost ou LightGBM) e técnicas de balanceamento de classes.
- Coletar mais lotes, principalmente de vinhos de alta qualidade, e validar o modelo com safras novas.